# Практическое задание 1. Вероятность дефолта по автокредиту

**Дисциплина «ИИ и машинное обучение»** · НИУ ВШЭ, Высшая школа бизнеса

Магистратура «Бизнес-аналитика и цифровые решения в финансах»

---

Банк выдаёт кредиты на покупку автомобиля. По каждой заявке известны
сумма кредита, цена машины, анкета заёмщика и его кредитная история из
бюро. Нужно с помощью **логистической регрессии** оценить вероятность
того, что заёмщик уйдёт в дефолт, и решить по каждой заявке из вашего
теста: **одобрить или отказать**.

У каждого свой вариант: свой набор заявок в скрытом тесте и своя
экономика. Номер варианта смотрите в файле `распределение_вариантов_по_студентам.xlsx` в нашем чате.

Что требуется, как считаются деньги портфеля и как ставится
оценка — в тексте задания. Здесь только данные, место под решение
и готовая сборка файла сдачи.

**Что сдать:** этот ноутбук `ФАМИЛИЯ_ИО.ipynb` с выполненными ячейками и файл
`ФАМИЛИЯ_ИО_predictions.csv` с результатами.

## Данные

Данные лежат на Яндекс Диске: https://disk.yandex.ru/d/wjsVUGFFIC2lCQ

Скачайте оттуда три файла и положите рядом с ноутбуком.

| Файл | Что внутри |
|---|---|
| `dataset.csv` | 60 000 заявок со столбцом `дефолт` — на них вы учитесь |
| `test_NN.csv` | 2 000 заявок вашего варианта, **без ответа** |
| `profit_loss.csv` | `доход_с_клиента` и `доля_возврата` всех вариантов — берите строку своего |

`NN` — номер вашего варианта двумя цифрами: `test_07.csv`, например.

В тесте 40 столбцов, в обучающем файле на один больше — `дефолт` (то, что хотим предсказывать).
Полное описание каждого столбца смотрите в [README.md](https://github.com/Rimest/ml_ai_course_gsb_hse/blob/main/homeworks/hw1_credit_default/README.md).

## Решение

Первая ячейка — преамбула: она фиксирует сид случайности. Выполните её
и подставляйте `SEED` везде, где есть параметр `random_state`. Ноутбук
должен при повторном запуске сверху вниз давать те же числа: результат должен быть воспроизводим.

In [ ]:
## ── воспроизводимость ──────────────────────────────────────────────
import random

import numpy as np

ФАМИЛИЯ_ИО = <ваша фамилия и инициалы> # Например, 'ИВАНОВ_ИИ'
SEED = <номер вашего варианта>
ВАРИАНТ = <номер вашего варианта>
random.seed(SEED)      # стандартный модуль random
np.random.seed(SEED)   # numpy

# Своего генератора у pandas и scikit-learn нет — оба берут случайность
# из numpy, так что строка выше покрывает и их. Но там, где параметр
# random_state есть, передавайте SEED явно: тогда результат не зависит
# ещё и от порядка, в котором выполнялись ячейки.

## Файл сдачи

Ячейка ниже собирает `ФАМИЛИЯ_ИО_predictions.csv`. Формат одинаков у всех,
поэтому код готовый — выполните ячейку, менять её не требуется.
`ФАМИЛИЯ_ИО` и `ВАРИАНТ` задаются в преамбуле, а кроме них к этому
моменту у вас должны быть три величины:

| Имя | Что это |
|---|---|
| `test` | таблица вашего теста, со столбцом `клиент` |
| `p_test` | вероятности дефолта по заявкам теста |
| `decisions` | ваши решения: 1 — одобрить, 0 — отказать |

Получится файл с четырьмя столбцами:

```
вариант,клиент,вероятность,одобрить
7,556231,0.365680,0
7,608914,0.154443,1
```

In [ ]:
## ── файл сдачи ─────────────────────────────────────────────────────
predictions = pd.DataFrame({
    "вариант": ВАРИАНТ,
    "клиент": test["клиент"].values,
    "вероятность": np.round(p_test, 6),
    "одобрить": np.asarray(decisions).astype(int),
})
predictions.to_csv(f"{ФАМИЛИЯ_ИО}_predictions.csv", index=False, encoding="utf-8")

## ── проверка перед сдачей ──────────────────────────────────────────
errors = []
if len(predictions) != 2000:
    errors.append("в файле %d строк, а в тесте 2 000 заявок" % len(predictions))
if predictions["клиент"].duplicated().any():
    errors.append("один и тот же клиент встречается несколько раз")
if not predictions["вероятность"].between(0, 1).all() or predictions["вероятность"].isna().any():
    errors.append("вероятности должны быть числами от 0 до 1 без пропусков")
if not set(predictions["одобрить"].unique()) <= {0, 1}:
    errors.append("столбец «одобрить» должен содержать только 0 и 1")
if not isinstance(ВАРИАНТ, int) or not 1 <= ВАРИАНТ <= 57:
    errors.append("номер варианта должен быть числом от 1 до 57")

print("Строк в файле: %d" % len(predictions))
print("Одобрено: %d (%.0f %%)"
      % (predictions["одобрить"].sum(), 100 * predictions["одобрить"].mean()))
print("Проверка: " + ("всё в порядке" if not errors else "; ".join(errors)))

predictions.head()

## Как оценивается работа

За работу — до 10 баллов, они складываются из четырёх частей.

| За что | Баллов | Как считается |
|---|---|---|
| Качество модели | 4 | разница вашего ROC-AUC с эталоном вашего варианта: не хуже чем на 0,005 — 4 балла; на 0,02 — 3; на 0,05 — 2; на 0,10 — 1; хуже — 0 |
| Деньги портфеля | 4 | ваши деньги в долях от денег эталона: от 0,95 — 4 балла; от 0,85 — 3; от 0,70 — 2; от 0,50 — 1; меньше — 0 |
| Формат сдачи | 1 | `ФАМИЛИЯ_ИО_predictions.csv` прошёл формальную проверку целиком |
| Ноутбук | 1 | приложен |

Качество и деньги считаются **раздельно**. Хорошо обученная модель ещё
не гарантирует денег: решения по заявкам — отдельная работа, и она
оценивается отдельно. Обратное тоже верно: удачные решения не спасут
слабую модель.

Эталон — решение курса, применённое к вашему же варианту. Варианты
неодинаковы по сложности, поэтому сравнение идёт с ним, а не
с фиксированной планкой.

Обоснование баллов не добавляет, но его отсутствие или неспособность
объяснить собственное решение снимает работу целиком.

## Обоснование

Дайте ответы на следующие вопросы:

1. **Что вы сделали с данными и почему.** Какие столбцы преобразовали,
   добавили или отбросили и что это дало на валидации.
2. **Как вы принимали решение одобрить или отказать.** Откуда взялось
   ваше правило и почему вы считаете его верным.
3. **Чего вы ждёте на тесте.** Какое качество и какие деньги, и почему
   результат может оказаться хуже, чем на валидации.